In [ ]:
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import cv2
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, confusion_matrix, matthews_corrcoef
from PIL import Image
from tqdm import tqdm
import os
import glob
import matplotlib.pyplot as plt

In [ ]:
class BinaryFocalLoss(nn.Module):
    def __init__(self, alpha=0.80, gamma=2.0, reduction='mean'):
        """
        Adaptado para lidar com saídas multiclasse [BATCH_SIZE, 2].
        alpha: Peso para a classe positiva (anomalia). Usamos 0.80.
               A classe 0 (normal) receberá o peso restante (0.20).
        gamma: Fator de foco.
        """
        super(BinaryFocalLoss, self).__init__()
        # Cria um tensor de pesos: [peso_normal, peso_anomalia] -> [0.20, 0.80]
        self.alpha_weight = torch.tensor([1.0 - alpha, alpha])
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        # inputs: [BATCH_SIZE, 2] (Logits brutos da rede para as 2 classes)
        # targets: [BATCH_SIZE] (Valores exatos da classe correta: 0 ou 1)
        
        # 1. Garante que o tensor de pesos esteja na mesma placa de vídeo que os dados
        weight = self.alpha_weight.to(inputs.device)
        
        # 2. Calcula a Cross Entropy padrão sem reduzir.
        # A função F.cross_entropy já sabe lidar perfeitamente com inputs [N, 2] e targets [N],
        # e já aplica os pesos do nosso tensor alpha automaticamente.
        ce_loss = F.cross_entropy(inputs, targets, weight=weight, reduction='none')
        
        # 3. pt é a probabilidade da classe correta. 
        # Matemática elegante: p_t = exp(-CE_loss) 
        pt = torch.exp(-ce_loss)
        
        # 4. Calcula o fator focal: (1 - pt)^gamma
        focal_term = (1 - pt) ** self.gamma
        
        # 5. Loss final (Note que alpha_t não está aqui pois já foi aplicado dentro da ce_loss)
        focal_loss = focal_term * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss
    
class VinDrMLODataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None):
        self.data = dataframe.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform
        
        # Mapeamento Binário (BI-RADS 1, 2, 3 = Benigno(0) | BI-RADS 4, 5 = Maligno(1))
        self.label_map = {
            'BI-RADS 1': 0, 
            'BI-RADS 2': 0, 
            'BI-RADS 3': 0, 
            'BI-RADS 4': 1, 
            'BI-RADS 5': 1
        }

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        
        # Caminho do DICOM
        img_path = f"{self.root_dir}/{row['study_id']}/{row['image_id']}.dicom"
        
        # Leitura e Normalização do DICOM
        ds = pydicom.dcmread(img_path)
        pixel_array = ds.pixel_array.astype(float)
        
        # Normalização Min-Max para a imagem médica
        pixel_array = (pixel_array - np.min(pixel_array)) / (np.max(pixel_array) - np.min(pixel_array))
        pixel_array_16bit = (pixel_array * 65535).astype(np.uint16)
        
        # Converte para imagem PIL em RGB (necessário para os pesos da EfficientNet)
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
        pixel_array_clahe = clahe.apply(pixel_array_16bit)

        pixel_array_8bit = (pixel_array_clahe / 256).astype(np.uint8)

        image = Image.fromarray(pixel_array_8bit).convert('RGB')
        
        lateralidade = row['laterality'] 
        
        # Espelha a mama direita para que todas fiquem orientadas como a esquerda
        if lateralidade == 'R':
            image = image.transpose(Image.FLIP_LEFT_RIGHT)
            
        # Pega a classe e converte para Binário
        label = self.label_map[row['breast_birads']]
        
        # Aplica Transformações (Tensor, Resize, Normalize...)
        if self.transform:
            image = self.transform(image)
            
        return image, torch.tensor(label, dtype=torch.long)

In [ ]:
def preview_clahe_from_folder(folder_path):
    """
    Lê todos os arquivos DICOM de uma pasta específica e plota
    o comparativo Sem CLAHE vs Com CLAHE.
    """
    # Busca todos os arquivos .dicom na pasta
    dicom_files = glob.glob(os.path.join(folder_path, "*.dicom"))
    
    if not dicom_files:
        print(f"Nenhum arquivo .dicom encontrado em: {folder_path}")
        return
        
    num_samples = len(dicom_files)
    fig, axes = plt.subplots(num_samples, 2, figsize=(10, 5 * num_samples))
    
    # Ajuste para caso seja apenas 1 imagem na pasta
    if num_samples == 1:
        axes = [axes]
        
    for i, img_path in enumerate(dicom_files):
        # Nome do arquivo para o título
        filename = os.path.basename(img_path)
        
        # Leitura do DICOM original
        ds = pydicom.dcmread(img_path)
        pixel_array = ds.pixel_array.astype(float)
        
        # --- 1. Processo Original (Sem CLAHE, apenas Min-Max para 8-bit) ---
        pixel_array_norm = (pixel_array - np.min(pixel_array)) / (np.max(pixel_array) - np.min(pixel_array))
        img_original_8bit = (pixel_array_norm * 255).astype(np.uint8)
        
        # --- 2. Novo Processo (Com CLAHE aplicado em 16-bit) ---
        pixel_array_16bit = (pixel_array_norm * 65535).astype(np.uint16)
        
        # Parâmetro de limite de contraste. Altere para testar (ex: 2.0, 3.0, 4.0)
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
        pixel_array_clahe = clahe.apply(pixel_array_16bit)
        
        # Redução para 8-bits para visualização
        img_clahe_8bit = (pixel_array_clahe / 256).astype(np.uint8)
        
        # --- Plotagem Lado a Lado ---
        axes[i][0].imshow(img_original_8bit, cmap='gray')
        axes[i][0].set_title(f"Original (Sem CLAHE)\n{filename}")
        axes[i][0].axis('off')
        
        axes[i][1].imshow(img_clahe_8bit, cmap='gray')
        axes[i][1].set_title("Com CLAHE (clipLimit=3.0)")
        axes[i][1].axis('off')
        
    plt.tight_layout()
    plt.show()

# Aponte para a pasta do estudo (study_id) dentro do seu repositório local
pasta_exame = "./c11a492c00de50377e2fddab5b1d935f" 
preview_clahe_from_folder(pasta_exame)
pasta_exame_2 = "./1caf7a7178dbc3fb9649b93ba42115a7"
preview_clahe_from_folder(pasta_exame_2)

In [ ]:
# --- Transformações (Atenção: NÃO há RandomHorizontalFlip aqui!) ---
train_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# --- Carregamento e Preparação do CSV ---
csv_path = "/backup/lucas/datasets/vindr-mammo/breast-level_annotations.csv"
images_dir = "/backup/lucas/datasets/vindr-mammo/images"

df_completo = pd.read_csv(csv_path)

# Filtra apenas MLO (Verifique se no CSV chama 'view' ou 'view_position')
df_mlo = df_completo[df_completo['view_position'] == 'MLO'].copy()

# Remove possíveis linhas sem BI-RADS anotado, se houver
df_mlo = df_mlo.dropna(subset=['breast_birads'])

# --- Split sem Data Leakage (por study_id) ---
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, temp_idx = next(gss1.split(df_mlo, groups=df_mlo['study_id']))

df_train = df_mlo.iloc[train_idx]
df_temp = df_mlo.iloc[temp_idx]

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
val_idx, test_idx = next(gss2.split(df_temp, groups=df_temp['study_id']))

df_val = df_temp.iloc[val_idx]
df_test = df_temp.iloc[test_idx]

print(f"Total Imagens MLO: {len(df_mlo)} | Treino: {len(df_train)} | Validação: {len(df_val)} | Teste: {len(df_test)}")

# --- DataLoaders ---
BATCH_SIZE = 8

train_dataset = VinDrMLODataset(dataframe=df_train, root_dir=images_dir, transform=train_transform)
val_dataset = VinDrMLODataset(dataframe=df_val, root_dir=images_dir, transform=val_transform)
test_dataset = VinDrMLODataset(dataframe=df_test, root_dir=images_dir, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Modelo ResNet50 ---
model = models.efficientnet_b4(weights='IMAGENET1K_V1')

for param in model.features[:4].parameters(): 
    param.requires_grad = False

# Troca a última camada para 2 classes (Benigno vs Maligno)
num_ftrs = model.classifier[1].in_features

model.classifier[1] = nn.Sequential(
    nn.Dropout(p=0.5),
    nn.Linear(num_ftrs, 2)
)
model = model.to(device)

# --- Pesos das Classes para o Desbalanceamento ---
# Casos malignos são minoria. Damos um peso maior para a Classe 1 (Maligno)
# Exemplo: Peso 1.0 para Benigno e 15.0 para Maligno. (Ajuste se precisar de mais sensibilidade)
weights = torch.tensor([1.0, 15.0]).to(device) 

criterion = BinaryFocalLoss(alpha=0.96, gamma=2.0)

# Optimizer com um Learning Rate baixo, ideal para transfer learning
optimizer = torch.optim.Adam(model.parameters(), lr=5e-5, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='min', 
    factor=0.5,
    patience=3,
)

In [ ]:
num_epochs = 50
best_auc = 0.0
best_mcc = 0.0
TRESHOLD = 0.50

for epoch in range(num_epochs):
    print(f"\n--- Época {epoch+1}/{num_epochs} ---")
    
    # ==================================
    # TREINAMENTO
    # ==================================
    model.train()
    train_loss = 0.0
    
    loop_treino = tqdm(train_loader, desc="Treinamento", leave=False)
    
    for images, labels in loop_treino:
        images = images.to(device)
        labels = labels.to(device).long()  
        
        optimizer.zero_grad()
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        loop_treino.set_postfix(loss=loss.item())
        
    train_loss = train_loss / len(train_loader.dataset)
    
    # ==================================
    # VALIDAÇÃO
    # ==================================
    model.eval()
    val_loss = 0.0
    
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        loop_val = tqdm(val_loader, desc="Validação", leave=False)
        
        for images, labels in loop_val:
            images = images.to(device)
            labels = labels.to(device).long()
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            
            probs = F.softmax(outputs, dim=1)[:, 1]
            
            preds = (probs >= TRESHOLD).long()
            
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    val_loss = val_loss / len(val_loader.dataset)
    
    # ==================================
    # MÉTRICAS
    # ==================================
    try:
        tn, fp, fn, tp = confusion_matrix(all_labels, all_preds, labels=[0, 1]).ravel()
    except ValueError:
        # caso raro: só uma classe presente
        tn = fp = fn = tp = 0
    
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        auc = 0.0

    #Calculating MCC (Matthews Correlation Coefficient)
    try:
        mcc = matthews_corrcoef(all_labels, all_preds)
    except ValueError:
        mcc = 0.0
    
    print(f"Loss Treino: {train_loss:.4f} | Loss Validação: {val_loss:.4f}")
    print(f"Matriz de Confusão -> TP:{tp} | FN:{fn} | TN:{tn} | FP:{fp}")
    print(f"Sensibilidade (Recall): {sensitivity:.4f}")
    print(f"Especificidade:       {specificity:.4f}")
    print(f"AUC-ROC:              {auc:.4f}")
    print(f"MCC:                  {mcc:.4f}")
    with open('log_treinamento_efficientnet.txt', 'a') as f:
        f.write(f"Epoca {epoch+1}/{num_epochs} | Loss T: {train_loss:.4f} | Loss V: {val_loss:.4f} | AUC: {auc:.4f} | MCC: {mcc:.4f}\n")
        f.write(f"Matriz de Confusão -> TP:{tp} | FN:{fn} | TN:{tn} | FP:{fp}\n")

    # ==================================
    # SALVAR MELHOR MODELO
    # ==================================
    scheduler.step(val_loss)

    #Salva o modelo com base na melhor AUC. Descartado devido ao desbalanceamento severo.
    #if auc > best_auc:
    #    best_auc = auc
    #    #torch.save(model.state_dict(), 'efficientnet_vindr_mlo_binario.pth')
    #    print(f"🔥 Novo melhor modelo salvo! (AUC: {best_auc:.4f})")
    
    if mcc > best_mcc:
        best_mcc = mcc
        torch.save(model.state_dict(), 'efficientnet_vindr_mlo_binario.pth')
        print(f"🔥 Novo melhor modelo salvo! (MCC: {best_mcc:.4f})")

In [ ]:
# ==================================
# AVALIAÇÃO FINAL NO CONJUNTO DE TESTE
# ==================================

# Carrega os pesos do melhor modelo salvo durante o treinamento
model.load_state_dict(torch.load('efficientnet_vindr_mlo_binario.pth'))
model.eval()

test_loss = 0.0
all_preds_test = []
all_labels_test = []
all_probs_test = []

with torch.no_grad():
    loop_test = tqdm(test_loader, desc="Teste Final", leave=False)
    
    for images, labels in loop_test:
        images = images.to(device)
        labels = labels.to(device).long()
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item() * images.size(0)
        
        probs = F.softmax(outputs, dim=1)[:, 1]
        preds = (probs >= TRESHOLD).long()
        
        all_labels_test.extend(labels.cpu().numpy())
        all_preds_test.extend(preds.cpu().numpy())
        all_probs_test.extend(probs.cpu().numpy())

test_loss = test_loss / len(test_loader.dataset)

# Cálculo das métricas finais
try:
    tn, fp, fn, tp = confusion_matrix(all_labels_test, all_preds_test, labels=[0, 1]).ravel()
except ValueError:
    tn = fp = fn = tp = 0

sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
auc = roc_auc_score(all_labels_test, all_probs_test) if len(np.unique(all_labels_test)) > 1 else 0.0
mcc = matthews_corrcoef(all_labels_test, all_preds_test) if len(np.unique(all_labels_test)) > 1 else 0.0

print("\n" + "="*40)
print("🏆 RESULTADOS FINAIS NO CONJUNTO DE TESTE 🏆")
print("="*40)
print(f"Loss Teste:           {test_loss:.4f}")
print(f"Matriz de Confusão -> TP:{tp} | FN:{fn} | TN:{tn} | FP:{fp}")
print(f"Sensibilidade (Recall): {sensitivity:.4f}")
print(f"Especificidade:       {specificity:.4f}")
print(f"AUC-ROC:              {auc:.4f}")
print(f"MCC:                  {mcc:.4f}")
print("="*40)